# 03 · LoRA, quantisation and preference optimisation

*Large Language Model Training*, notebook 3 of 4 · lessons 4 and 5: **Adapt efficiently with LoRA and QLoRA** and **Understand preference optimization and its data**

Fine-tuning every weight works, but it needs memory for every gradient and optimiser state, and a full copy of the model for every task. **LoRA** freezes the base model and learns a small low-rank update beside each chosen weight matrix; **QLoRA** also stores the frozen base in a few bits per weight. In the first half of this notebook you build a LoRA layer, teach the instruction model from notebook 02 a new instruction through adapters alone, prove that the base did not change, merge the adapter, and quantise the frozen weights. In the second half you turn from imitating answers to *preferring* one answer over another: you implement the **DPO** loss, see what it rewards, and watch a shortcut in preference data become a learned preference. The models are tiny and the tasks are symbol games, so everything here shows mechanics, not the behaviour of large models.

## What you will build

- `lora_forward(x, W, A, B, alpha)`: a frozen weight plus a trainable low-rank update;
- `lora_parameter_count(shapes, rank)`: how many parameters a set of adapters trains;
- `changed_weights(before, after)`: proof that training left the frozen base untouched;
- `merge_lora(W, A, B, alpha)`: one matrix that gives the same outputs as base plus adapter;
- `quantise_int8(W)` and `dequantise(q, scales)`: 8-bit weights with one scale per output column;
- `sequence_logprob(log_probs, targets, mask)` and `dpo_loss(...)`: the DPO objective computed from response log-probabilities.

## Learning objectives

By the end of this notebook you will be able to:

1. implement a LoRA layer, explain why one factor starts at zero, and count adapter parameters;
2. verify that adapter training leaves the base bit for bit unchanged, and that merging gives the same outputs up to rounding;
3. quantise weights to 8 bits, measure the round-trip error and the memory saved, and explain why QLoRA trains the adapter on top of the quantised base;
4. compute the DPO loss from chosen and rejected log-probabilities with a frozen reference, and explain what it rewards;
5. show how a shortcut in preference data, such as length, becomes a learned preference, and how counter-examples remove it.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for one plot. Nothing else is imported, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Nothing is downloaded: the data are generated in the notebook from fixed seeds, so your numbers should match the ones quoted in the text up to small differences between computers.
- Two hidden recap cells hold the tiny tensor engine from Base Model Training and the pieces from notebooks 01 and 02.
- The notebook runs in a few seconds on a laptop, most of it in four short training runs; in the browser it takes longer.

## Setup

Run this cell first. It imports NumPy and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you write against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import time

import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below is the reverse-mode autograd engine from Base Model Training, pasted unchanged: `Tensor` (with `+`, `-`, `*`, `/`, `**`, `@`, `exp`, `log`, `tanh`, `relu`, `sigmoid`, `sum`, `mean`, `reshape`, `transpose`, indexing, `log_softmax`, `softmax` and `backward`), `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it was checked against numerical gradients. Run the cell; open it only if you want to look back at the code.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 📦 Recap: what you built earlier

The second hidden cell re-defines the pieces from notebooks 01 and 02, under the same names:

- from notebook 01: the tokeniser (`VOCAB`, `encode`, `decode`, `PAD`, `END`, …), the toy world with the 40 reserved `EVAL_STRINGS`, the tiny transformer `init_tiny_gpt` / `gpt_forward`, `Adam`, and `pretrain_base(steps, seed)`;
- from notebook 02: `render_chat`, `assistant_flags`, `response_only_targets`, the instructions `TASKS` with `OLD_TASKS = ["copy", "reverse", "first", "last"]`, `task_messages`, `pad_rows`, `sft_batch`, `sft_train`, `generate` and `task_accuracy`.

Two things are new. `TASKS` now also contains **next**, which shifts every letter one place along the alphabet (`next abc` → `bcd`, and `z` wraps round to `a`); the instruction model has never seen it. And `gpt_forward(params, ids, adapters)` applies every weight matrix through `project(x, params, name, adapters)`, which calls **your** `lora_forward` from section 1 whenever an adapter is attached to that matrix. `sft_train(..., adapters=adapters)` trains only the adapter matrices.

In [ ]:
# 📦 Recap: the pieces from notebooks 01 and 02, under their usual names.

# ── Vocabulary and tokeniser (notebook 01) ─────────────────────────────────
# One token per character, plus five special tokens. The chat template's
# tokens are reserved from the start, as real tokenisers reserve theirs.
SPECIALS = ["<pad>", "<eos>", "<|user|>", "<|assistant|>", "<|end|>"]
VOCAB = SPECIALS + list(" .,:?0123456789abcdefghijklmnopqrstuvwxyz")
TOKEN_ID = {token: index for index, token in enumerate(VOCAB)}
PAD, EOS, USER, ASSISTANT, END = range(len(SPECIALS))


def encode(text):
    """Text -> list of token ids. Each special token, such as <|end|>, becomes a single id."""
    ids, i = [], 0
    while i < len(text):
        special = next((s for s in SPECIALS if text.startswith(s, i)), None)
        if special is not None:
            ids.append(TOKEN_ID[special])
            i += len(special)
        else:
            ids.append(TOKEN_ID[text[i]])
            i += 1
    return ids


def decode(ids):
    """List of token ids -> text."""
    return "".join(VOCAB[int(i)] for i in ids)


# ── The toy world: three sources of generated text (notebook 01) ────────────
LETTERS = "abcdefghijklmnopqrstuvwxyz"
ANIMALS = ["fox", "owl", "cat", "hen", "elk", "bee", "yak", "eel", "ram", "doe"]
COLOURS = ["red", "tan", "grey", "brown", "white", "black", "gold"]
VERBS = ["ran", "sat", "slept", "hid", "sang", "swam", "ate"]
PREPOSITIONS = ["to", "by", "near", "in"]
PLACES = ["river", "forest", "hill", "barn", "pond", "field", "lake"]
ITEMS = ["figs", "plums", "nuts", "pears", "eggs", "beans", "limes"]
NUMBERS = ["one", "two", "three", "four", "five", "six"]


def evaluation_strings(count=40, seed=123):
    """The reserved three-letter strings of the held-out evaluation set, sorted."""
    picker = np.random.default_rng(seed)
    chosen = set()
    while len(chosen) < count:
        chosen.add("".join(picker.choice(list(LETTERS), 3)))
    return sorted(chosen)


EVAL_STRINGS = evaluation_strings()
RESERVED = set(EVAL_STRINGS) | {s[::-1] for s in EVAL_STRINGS}  # no training text may contain these


def random_string(picker, banned=()):
    """A random three-letter string that is not in `banned`."""
    while True:
        s = "".join(picker.choice(list(LETTERS), 3))
        if s not in banned:
            return s


def make_document(source, picker, banned=()):
    """One generated document from 'stories', 'letters' or 'lists'."""
    if source == "stories":
        return " ".join(
            f"the {picker.choice(COLOURS)} {picker.choice(ANIMALS)} {picker.choice(VERBS)} "
            f"{picker.choice(PREPOSITIONS)} the {picker.choice(PLACES)}."
            for _ in range(picker.integers(2, 5))
        )
    if source == "letters":
        sentences = []
        for _ in range(picker.integers(2, 5)):
            s = random_string(picker, banned)
            sentences.append([f"{s} backwards is {s[::-1]}.", f"{s} starts with {s[0]}.",
                              f"{s} ends with {s[-1]}.", f"say {s} twice: {s} {s}."][picker.integers(4)])
        return " ".join(sentences)
    picks = picker.choice(ITEMS, picker.integers(2, 4), replace=False)
    return "list: " + ", ".join(f"{picker.choice(NUMBERS)} {item}" for item in picks) + "."


def make_clean_corpus(n_docs=500, seed=7):
    """Generated documents with ids, sources and text. Letter games never use a reserved string."""
    picker = np.random.default_rng(seed)
    docs = []
    for i in range(n_docs):
        source = str(picker.choice(["stories", "letters", "lists"], p=[0.45, 0.4, 0.15]))
        docs.append({"doc_id": f"d{i:04d}", "source": source,
                     "text": make_document(source, picker, RESERVED)})
    return docs


def project(x, params, name, adapters=None):
    """x @ W for the named weight matrix, plus its LoRA update when an adapter is attached to it."""
    if adapters and name in adapters:
        A, B, alpha = adapters[name]
        return lora_forward(x, params[name], A, B, alpha)
    return x @ params[name]


# ── A tiny causal transformer, written with the tensor engine ───────────────
# ids (batch, T) -> embeddings (batch, T, D) -> logits (batch, T, len(VOCAB)).
# Each block is pre-norm: x + attention(layer_norm(x)), then x + mlp(layer_norm(x)).
CONTEXT, D_MODEL, N_HEADS, N_LAYERS, D_FF = 32, 32, 2, 2, 128


def init_tiny_gpt(seed=0):
    """Every weight of the model, as a dict name -> trainable Tensor."""
    init = np.random.default_rng(seed)
    D = D_MODEL
    params = {"tok": parameter(len(VOCAB), D, rng=init, scale=0.1),
              "pos": parameter(CONTEXT, D, rng=init, scale=0.1)}
    for b in range(N_LAYERS):
        params[f"b{b}.ln1_g"], params[f"b{b}.ln1_b"] = Tensor(np.ones(D), requires_grad=True), zeros_parameter(D)
        for name in ("Wq", "Wk", "Wv"):
            params[f"b{b}.{name}"] = parameter(D, D, rng=init, scale=D ** -0.5)
        params[f"b{b}.Wo"] = parameter(D, D, rng=init, scale=(2 * N_LAYERS * D) ** -0.5)
        params[f"b{b}.ln2_g"], params[f"b{b}.ln2_b"] = Tensor(np.ones(D), requires_grad=True), zeros_parameter(D)
        params[f"b{b}.W1"], params[f"b{b}.b1"] = parameter(D, D_FF, rng=init, scale=D ** -0.5), zeros_parameter(D_FF)
        params[f"b{b}.W2"] = parameter(D_FF, D, rng=init, scale=(2 * N_LAYERS * D_FF) ** -0.5)
        params[f"b{b}.b2"] = zeros_parameter(D)
    params["lnf_g"], params["lnf_b"] = Tensor(np.ones(D), requires_grad=True), zeros_parameter(D)
    params["Wout"] = parameter(D, len(VOCAB), rng=init, scale=D ** -0.5)
    return params


def gpt_forward(params, ids, adapters=None):
    """Next-token logits (batch, T, len(VOCAB)) for integer token ids (batch, T), with T <= CONTEXT."""
    ids = np.asarray(ids)
    batch, T = ids.shape
    D = params["tok"].shape[1]
    hd = D // N_HEADS
    future = np.triu(np.full((T, T), -1e9), 1)  # position t may not look at positions after t
    x = params["tok"][ids] + params["pos"][np.arange(T)]
    for b in range(N_LAYERS):
        h = layer_norm(x, params[f"b{b}.ln1_g"], params[f"b{b}.ln1_b"])
        q, k, v = (project(h, params, f"b{b}.{name}", adapters).reshape(batch, T, N_HEADS, hd).transpose(0, 2, 1, 3)
                   for name in ("Wq", "Wk", "Wv"))
        attention = ((q @ k.transpose(0, 1, 3, 2)) * hd ** -0.5 + future).softmax(axis=-1)
        mixed = (attention @ v).transpose(0, 2, 1, 3).reshape(batch, T, D)
        x = x + project(mixed, params, f"b{b}.Wo", adapters)
        h = layer_norm(x, params[f"b{b}.ln2_g"], params[f"b{b}.ln2_b"])
        hidden = (project(h, params, f"b{b}.W1", adapters) + params[f"b{b}.b1"]).relu()
        x = x + project(hidden, params, f"b{b}.W2", adapters) + params[f"b{b}.b2"]
    return project(layer_norm(x, params["lnf_g"], params["lnf_b"]), params, "Wout", adapters)


class Adam:
    """Adam with bias correction and clipping of the global gradient norm, for a list of Tensors."""

    def __init__(self, tensors, lr=3e-3, betas=(0.9, 0.99), eps=1e-8, clip=1.0):
        self.tensors, self.lr, self.betas, self.eps, self.clip = list(tensors), lr, betas, eps, clip
        self.m = [np.zeros_like(t.data) for t in self.tensors]
        self.v = [np.zeros_like(t.data) for t in self.tensors]
        self.t = 0

    def step(self):
        self.t += 1
        b1, b2 = self.betas
        grads = [np.zeros_like(t.data) if t.grad is None else t.grad for t in self.tensors]
        norm = float(np.sqrt(sum((g ** 2).sum() for g in grads)))
        shrink = min(1.0, self.clip / (norm + 1e-12))
        for t, g, m, v in zip(self.tensors, grads, self.m, self.v):
            g = g * shrink
            m[...] = b1 * m + (1 - b1) * g
            v[...] = b2 * v + (1 - b2) * g * g
            t.data -= self.lr * (m / (1 - b1 ** self.t)) / (np.sqrt(v / (1 - b2 ** self.t)) + self.eps)
        return norm


# ── Pretraining (notebook 01) ───────────────────────────────────────────────
def pack_documents(docs):
    """One token stream: every document followed by <eos>."""
    stream = []
    for doc in docs:
        stream += encode(doc["text"]) + [EOS]
    return np.array(stream)


def pretrain_base(steps=150, seed=0, batch_size=16, lr=3e-3):
    """A shorter run of notebook 01's recipe: random windows of the packed clean corpus."""
    stream = pack_documents(make_clean_corpus())
    params = init_tiny_gpt(seed)
    optimiser = Adam(list(params.values()), lr=lr)
    window_picker = np.random.default_rng(seed)
    for _ in range(steps):
        starts = window_picker.integers(0, len(stream) - CONTEXT - 1, batch_size)
        windows = np.stack([stream[s:s + CONTEXT + 1] for s in starts])
        loss = cross_entropy(gpt_forward(params, windows[:, :-1]), windows[:, 1:])
        zero_grad(optimiser.tensors)
        loss.backward()
        optimiser.step()
    return params


# ── Chat template and response-only targets (notebook 02) ──────────────────
def render_chat(messages, add_generation_prompt=False):
    """Messages -> one string: <|role|>content<|end|> per message, then <|assistant|> if asked."""
    text = "".join(f"<|{m['role']}|>{m['content']}<|end|>" for m in messages)
    return text + ("<|assistant|>" if add_generation_prompt else "")


def assistant_flags(messages):
    """One flag per token of encode(render_chat(messages)): 1 for assistant content and its <|end|>."""
    flags = []
    for m in messages:
        scored = 1 if m["role"] == "assistant" else 0
        flags += [0] + [scored] * len(encode(m["content"])) + [scored]
    return flags


def response_only_targets(ids, flags):
    """Shift once: inputs = ids[:-1], targets = ids[1:], and the mask follows the targets."""
    ids, flags = np.asarray(ids), np.asarray(flags, dtype=float)
    return ids[:-1], ids[1:], flags[1:]


# ── Instruction tasks (notebook 02; the task "next" is new in notebook 03) ──
TASKS = {
    "copy": lambda s: s,
    "reverse": lambda s: s[::-1],
    "first": lambda s: s[0],
    "last": lambda s: s[-1],
    "next": lambda s: "".join(LETTERS[(LETTERS.index(c) + 1) % 26] for c in s),
}
OLD_TASKS = ["copy", "reverse", "first", "last"]


def task_messages(task, s, with_answer=True):
    """A one-turn conversation asking for `task` on the string s."""
    messages = [{"role": "user", "content": f"{task} {s}"}]
    if with_answer:
        messages.append({"role": "assistant", "content": TASKS[task](s)})
    return messages


def pad_rows(rows):
    """Stack (inputs, targets, mask) rows of different lengths; padding gets <pad> and mask 0."""
    T = max(len(inputs) for inputs, _, _ in rows)
    inputs, targets, mask = np.full((len(rows), T), PAD), np.full((len(rows), T), PAD), np.zeros((len(rows), T))
    for i, (x, y, m) in enumerate(rows):
        inputs[i, :len(x)], targets[i, :len(y)], mask[i, :len(m)] = x, y, m
    return inputs, targets, mask


def sft_batch(tasks, batch_size, seed, task_weights=None):
    """A padded batch of one-turn conversations on training strings (never the evaluation strings)."""
    picker = np.random.default_rng(seed)
    rows = []
    for _ in range(batch_size):
        task = str(picker.choice(tasks, p=task_weights))
        messages = task_messages(task, random_string(picker, RESERVED))
        rows.append(response_only_targets(encode(render_chat(messages)), assistant_flags(messages)))
    return pad_rows(rows)


def sft_train(params, tasks, steps, seed=0, lr=3e-3, batch_size=32, task_weights=None,
              score_prompts=False, adapters=None):
    """Supervised fine-tuning; returns the loss of every step. With adapters, only A and B are trained."""
    tensors = [t for A, B, _ in adapters.values() for t in (A, B)] if adapters else list(params.values())
    optimiser = Adam(tensors, lr=lr)
    losses = []
    for step in range(steps):
        inputs, targets, mask = sft_batch(tasks, batch_size, seed * 100_000 + step, task_weights)
        if score_prompts:  # the variant without the response-only mask
            mask = (targets != PAD).astype(float)
        loss = cross_entropy(gpt_forward(params, inputs, adapters), targets, mask)
        zero_grad(tensors)
        loss.backward()
        optimiser.step()
        losses.append(float(loss.data))
    return losses


def generate(params, prompts, adapters=None, max_new_tokens=5):
    """Greedy decoding until <|end|>. Prompts of the same token length are decoded together."""
    encoded = [encode(p) for p in prompts]
    answers = [None] * len(prompts)
    for length in sorted({len(ids) for ids in encoded}):
        group = [i for i, ids in enumerate(encoded) if len(ids) == length]
        tokens = np.array([encoded[i] for i in group])
        out, done = [[] for _ in group], np.zeros(len(group), dtype=bool)
        for _ in range(max_new_tokens):
            next_ids = gpt_forward(params, tokens, adapters).data[:, -1].argmax(axis=-1)
            for j, token in enumerate(next_ids):
                if not done[j] and token == END:
                    done[j] = True
                elif not done[j]:
                    out[j].append(token)
            if done.all():
                break
            tokens = np.concatenate([tokens, next_ids[:, None]], axis=1)
        for j, i in enumerate(group):
            answers[i] = decode(out[j])
    return answers


def task_accuracy(params, tasks, strings=EVAL_STRINGS, adapters=None):
    """Exact-match accuracy of greedy answers on the given strings, per task."""
    scores = {}
    for task in tasks:
        prompts = [render_chat(task_messages(task, s, with_answer=False), add_generation_prompt=True) for s in strings]
        answers = generate(params, prompts, adapters)
        scores[task] = float(np.mean([a == TASKS[task](s) for a, s in zip(answers, strings)]))
    return scores

## 0 · Re-create the instruction model

The cell below repeats notebook 02's recipe exactly (150 pretraining steps, then 250 steps of replies-only fine-tuning on the four instructions), so it produces the same model. It takes a couple of seconds.

In [ ]:
start = time.perf_counter()
instruct = pretrain_base(steps=150, seed=0)
sft_train(instruct, OLD_TASKS, steps=250, seed=0)
print(f"re-created notebook 02's instruction model in {time.perf_counter() - start:.1f} s")
print("accuracy on the reserved strings:", task_accuracy(instruct, OLD_TASKS + ["next"]))

**What to notice.** The model follows the four instructions it was trained on and fails the new one: `next` asks for something it has never been shown. The rest of the first half teaches it `next` without changing any of its weights.

## 1 · The LoRA layer

Our layers compute $xW$, with $x$ of shape `(…, d_in)` and $W$ of shape `(d_in, d_out)`. **LoRA** keeps $W$ frozen and adds a trainable update $\Delta W = AB$ built from two thin matrices, $A$ of shape `(d_in, r)` and $B$ of shape `(r, d_out)`, with a rank $r$ much smaller than `d_in` and `d_out`:

$$h = xW + \frac{\alpha}{r}\,(xA)B.$$

- **The update has rank at most $r$.** It can only change the layer's outputs within an $r$-dimensional subspace, which is often enough to adapt a pretrained model to a narrow task.
- **It is cheap.** A full update of $W$ has $d_\text{in} d_\text{out}$ parameters; the adapter has $r(d_\text{in} + d_\text{out})$. For a $4096 \times 4096$ projection at rank 8 that is 65,536 instead of 16,777,216.
- **$B$ starts at zero, $A$ at random.** Then $\Delta W = 0$, so the adapted model starts *exactly* where the base is, and the gradient with respect to $B$, which is $(xA)^\top g$ for an upstream gradient $g$, is not zero, so training can begin.
- **The scale $\alpha / r$** keeps the size of the update roughly constant when you change $r$, so a learning rate tuned at one rank still works at another.

The LoRA paper multiplies column vectors and writes $h = W_0 x + BAx$ with $B$ of shape $d \times r$ and $A$ of shape $r \times k$. Our row-vector $A$ and $B$ are the transposes of the paper's, so our $AB$ is the transpose of its $BA$: the same update, written the other way round. In both, $B$ is the factor on the output side and starts at zero.

In [ ]:
demo_picker = np.random.default_rng(0)
A_demo, B_demo = demo_picker.normal(size=(32, 2)), demo_picker.normal(size=(2, 128))
print("A @ B has shape", (A_demo @ B_demo).shape, "and rank", np.linalg.matrix_rank(A_demo @ B_demo))
print("parameters: full 32 x 128 update", 32 * 128, "| rank-2 factors", 2 * (32 + 128))

### ✍️ Your turn: `lora_forward`

Write `lora_forward(x, W, A, B, alpha)`, which returns $xW + \frac{\alpha}{r}(xA)B$ with $r$ = `A.shape[1]`.

- Shapes: `x` `(…, d_in)`, `W` `(d_in, d_out)`, `A` `(d_in, r)`, `B` `(r, d_out)`; the result has shape `(…, d_out)`.
- Use only `@`, `*` and `+`, so that the same function works on NumPy arrays (as in the check) and on the tensor engine's `Tensor`s (inside `gpt_forward`).

Hint: compute `x @ A` first; it is only `r` columns wide, so `(x @ A) @ B` never builds the full `d_in × d_out` update.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference is in the collapsed cell below; open it only when you are stuck.

In [ ]:
def lora_forward_reference(x, W, A, B, alpha):
    return x @ W + (x @ A) @ B * (alpha / A.shape[1])

In [ ]:
def lora_forward(x, W, A, B, alpha):
    """x @ W + (alpha / r) * (x @ A) @ B with r = A.shape[1]. x: (..., d_in), W: (d_in, d_out), A: (d_in, r), B: (r, d_out)."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
case_picker = np.random.default_rng(1)
x_small, W_small = case_picker.normal(size=(3, 4)), case_picker.normal(size=(4, 5))
A_small, B_small = case_picker.normal(size=(4, 2)), case_picker.normal(size=(2, 5))
lora_forward = practice("lora_forward", lora_forward, lora_forward_reference,
                        cases=[(x_small, W_small, A_small, B_small, 4.0),
                               (x_small, W_small, A_small, np.zeros((2, 5)), 4.0),
                               (case_picker.normal(size=(2, 3, 4)), W_small, A_small[:, :1], B_small[:1], 1.0)])

**Expected behaviour.** With `B` all zeros, the output equals `x @ W` exactly, which is why a freshly attached adapter changes nothing. The third case has a batch and a time axis, `(2, 3, 4)`, as the model's activations do, and a rank-1 adapter.

### ✍️ Your turn: `lora_parameter_count`

Write `lora_parameter_count(shapes, rank)`, where `shapes` is a list of `(d_in, d_out)` pairs, one for each adapted matrix. Return the total number of adapter parameters, $\sum r\,(d_\text{in} + d_\text{out})$, as an integer.

*Type it rather than pasting it: typing is how the shapes and indices sink in.*

In [ ]:
def lora_parameter_count_reference(shapes, rank):
    return int(sum(rank * (d_in + d_out) for d_in, d_out in shapes))

In [ ]:
def lora_parameter_count(shapes, rank):
    """Sum of rank * (d_in + d_out) over the (d_in, d_out) shapes of the adapted matrices."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
lora_parameter_count = practice("lora_parameter_count", lora_parameter_count, lora_parameter_count_reference,
                                cases=[([(4096, 4096)], 8), ([(32, 32)] * 4 + [(32, 128), (128, 32)], 4), ([], 4)])
print("4096 x 4096 at rank 8:", lora_parameter_count([(4096, 4096)], 8), "adapter parameters vs", 4096 * 4096)

We attach adapters to every projection of both blocks and to the output layer: `Wq`, `Wk`, `Wv`, `Wo`, `W1` and `W2` in `b0` and `b1`, and `Wout`. Count what that trains.

In [ ]:
LORA_TARGETS = [f"b{b}.{name}" for b in range(N_LAYERS) for name in ("Wq", "Wk", "Wv", "Wo", "W1", "W2")] + ["Wout"]
n_base = sum(t.data.size for t in instruct.values())
for r in (1, 4, 16):
    n_adapter = lora_parameter_count([instruct[name].shape for name in LORA_TARGETS], r)
    print(f"rank {r:>2}: {n_adapter:>6,} adapter parameters = {n_adapter / n_base:6.1%} of the {n_base:,} base parameters")

**What to notice.** At rank 4 the adapters hold about a sixth as many parameters as the base. That fraction looks large only because our matrices are tiny: the fraction of a $d \times d$ matrix is $2r/d$, which is 25% at $d = 32$ but 0.2% at $d = 4096$. This is a parameter comparison only. Training memory also holds activations, which LoRA does not shrink, so the total saving is smaller than the parameter ratio suggests.

## 2 · Teach a new instruction through adapters alone

`frozen_copy` copies the instruction model into tensors that do not require gradients: the frozen base. `init_adapters` creates one `(A, B, alpha)` triple for each target matrix, with $A \sim \mathcal{N}(0, 1/d_\text{in})$ and $B = 0$, and we set $\alpha = 2r$. Before training, the adapted model must give **bit for bit** the same logits as the base; the cell checks that.

The training data mix the new instruction with the old ones: half the conversations ask for `next`, and the other half are spread evenly over `copy`, `reverse`, `first` and `last`. Mixing old examples back in, often called **replay**, protects behaviour you want to keep (notebook 04 measures what happens without it). Training runs 150 steps with Adam, which updates only the adapter matrices.

In [ ]:
def frozen_copy(params):
    """A copy of every weight as a Tensor that does not require gradients."""
    return {name: Tensor(t.data.copy()) for name, t in params.items()}


def init_adapters(params, targets, rank, alpha, seed):
    """A: (d_in, r) ~ N(0, 1/d_in) and B: (r, d_out) = 0, so the adapted model starts identical to the base."""
    picker = np.random.default_rng(seed)
    adapters = {}
    for name in targets:
        d_in, d_out = params[name].shape
        adapters[name] = (Tensor(picker.normal(0.0, d_in ** -0.5, (d_in, rank)), requires_grad=True),
                          Tensor(np.zeros((rank, d_out)), requires_grad=True), alpha)
    return adapters


rank = 4            # 🧪 try 1 and 16, then run this cell and the next ones again
replay_share = 0.5  # 🧪 the share of old instructions in the adapter's data; try 0.0

base_frozen = frozen_copy(instruct)
before = {name: t.data.copy() for name, t in base_frozen.items()}  # a snapshot of the frozen base
adapters = init_adapters(base_frozen, LORA_TARGETS, rank=rank, alpha=2 * rank, seed=0)

probe_ids = sft_batch(OLD_TASKS + ["next"], batch_size=8, seed=99)[0]
assert np.array_equal(gpt_forward(base_frozen, probe_ids, adapters).data, gpt_forward(base_frozen, probe_ids).data)
print("before training, base + adapters gives exactly the same logits as the base")

task_weights = [1 - replay_share] + [replay_share / len(OLD_TASKS)] * len(OLD_TASKS)
start = time.perf_counter()
adapter_losses = sft_train(base_frozen, ["next"] + OLD_TASKS, steps=150, seed=10, lr=5e-3,
                           task_weights=task_weights, adapters=adapters)
print(f"trained the rank-{rank} adapters for 150 steps in {time.perf_counter() - start:.1f} s")
accuracy_base = task_accuracy(base_frozen, OLD_TASKS + ["next"])
accuracy_adapted = task_accuracy(base_frozen, OLD_TASKS + ["next"], adapters=adapters)
print("base:          ", accuracy_base)
print("base + adapter:", accuracy_adapted)

**What to notice.** With the adapters switched on, the model answers `next` on strings it never saw, and it still follows the old instructions, perhaps with a few new mistakes. With the adapters switched off (`adapters=None`) it is exactly the instruction model again: an adapter is an *add-on*, and one base can carry several of them, swapped per request. Try `replay_share = 0.0` to see what training on the new instruction alone does to the old ones.

But did the base really stay frozen? Do not take it on trust: compare every weight with the snapshot taken before training.

### ✍️ Your turn: `changed_weights`

Write `changed_weights(before, after)`. Both arguments are dicts that map a weight name to a NumPy array. Return the **sorted list** of the names whose arrays are not exactly equal, with `np.array_equal`, which compares shapes and every element.

*Type it rather than pasting it: typing is how the shapes and indices sink in.*

In [ ]:
def changed_weights_reference(before, after):
    return sorted(name for name in before if not np.array_equal(before[name], after[name]))

In [ ]:
def changed_weights(before, after):
    """Sorted names whose arrays differ between the dicts before and after (exact comparison)."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
same = {"a": np.ones(3), "b": np.zeros((2, 2))}
nudged = {"a": np.ones(3), "b": np.array([[0.0, 1e-12], [0.0, 0.0]])}
changed_weights = practice("changed_weights", changed_weights, changed_weights_reference,
                           cases=[(same, same), (same, nudged), (nudged, same)])
after = {name: t.data for name, t in base_frozen.items()}
print("base weights changed by adapter training:", changed_weights(before, after))

**Expected behaviour.** A change of $10^{-12}$ in one element counts: "frozen" means bit for bit, not approximately. For the real model the list is empty. The frozen tensors never required gradients, and the optimiser only ever received the adapter matrices.

## 3 · Merge the adapter into the weights

At inference time the extra matrix products of an adapter cost a little time. If you serve one task, you can fold each adapter into its weight once:

$$W' = W + \frac{\alpha}{r} AB, \qquad xW' = xW + \frac{\alpha}{r}(xA)B.$$

The two sides are equal in exact arithmetic, but not bit for bit in floating point, because the additions happen in a different order and each one rounds. Expect differences around $10^{-15}$ and compare with a tolerance. A merged model is a **new artefact**: if you merge and then quantise, or merge into a quantised base, the rounding is different again, so evaluate the exact artefact you ship.

### ✍️ Your turn: `merge_lora`

Write `merge_lora(W, A, B, alpha)`, which returns $W + \frac{\alpha}{r}AB$ with $r$ = `A.shape[1]`, as a NumPy array of the same shape as `W`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.*

In [ ]:
def merge_lora_reference(W, A, B, alpha):
    return W + (alpha / A.shape[1]) * (A @ B)

In [ ]:
def merge_lora(W, A, B, alpha):
    """W + (alpha / r) * A @ B, with r = A.shape[1]."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
merge_lora = practice("merge_lora", merge_lora, merge_lora_reference,
                      cases=[(W_small, A_small, B_small, 4.0), (W_small, A_small, np.zeros((2, 5)), 4.0)])

merged = {name: Tensor(t.data.copy()) for name, t in base_frozen.items()}
for name, (A, B, alpha) in adapters.items():
    merged[name] = Tensor(merge_lora(base_frozen[name].data, A.data, B.data, alpha))
logits_adapter = gpt_forward(base_frozen, probe_ids, adapters).data
logits_merged = gpt_forward(merged, probe_ids).data
print("largest difference in logits, merged vs base + adapters:", np.abs(logits_merged - logits_adapter).max())
print("bit-for-bit equal:", np.array_equal(logits_merged, logits_adapter))
prompts = [render_chat(task_messages("next", s, with_answer=False), add_generation_prompt=True) for s in EVAL_STRINGS]
same_answers = generate(merged, prompts) == generate(base_frozen, prompts, adapters)
print("identical greedy answers on the 40 'next' prompts:", same_answers)

## 4 · Store the frozen base in 8 bits

A frozen base is only read, never updated, so it can be stored in fewer bits. Memory per parameter is 8 bytes in float64 (NumPy's default, which we use), 4 in float32, 2 in bfloat16 and 1 in int8.

**Symmetric absmax quantisation**, one scale per output column $j$:

$$s_j = \frac{\max_i |W_{ij}|}{127}, \qquad q_{ij} = \operatorname{round}\!\left(\frac{W_{ij}}{s_j}\right) \in \{-127, \dots, 127\}, \qquad \hat W_{ij} = q_{ij}\, s_j.$$

The integers $q$ are stored as int8, with one float scale per column, and $\hat W$ is rebuilt when the layer is used. Rounding moves each weight by at most half a step, $s_j / 2$, so the largest error relative to the column's largest weight is $1/254 \approx 0.4\%$. A column of zeros would give $s_j = 0$ and a division by zero; give it the scale 1 instead, which stores it exactly.

Per-column scales matter because columns differ in size, and one large weight would otherwise waste the levels of every other column. Real methods go further. LLM.int8() keeps rare outlier features in 16 bits; QLoRA stores the frozen base in a 4-bit *NormalFloat* format whose levels suit normally distributed weights, also quantises the scales ("double quantisation"), and pages optimiser states to save memory. Quantised storage does not mean every operation runs in 4 bits: the weights are dequantised for each product, while the adapter and its gradients stay in higher precision.

In [ ]:
w_demo = np.array([0.5, -1.27, 0.02, 0.0])
s_demo = np.abs(w_demo).max() / 127
q_demo = np.round(w_demo / s_demo).astype(np.int8)
print("scale", s_demo, "| integers", q_demo, "| rebuilt", q_demo * s_demo, "| error", np.abs(q_demo * s_demo - w_demo))

### ✍️ Your turn: `quantise_int8` and `dequantise`

Write `quantise_int8(W)` for a 2-D array `W` of shape `(d_in, d_out)`. Return a tuple `(q, scales)`:

- `scales`: shape `(d_out,)`, the largest absolute value in each column divided by 127, with 1.0 for a column that is all zeros;
- `q`: shape `(d_in, d_out)`, `np.round(W / scales)`, clipped to $[-127, 127]$ and converted with `.astype(np.int8)`.

Then write `dequantise(q, scales)`, which returns `q` as floats multiplied by `scales` (broadcast over the rows).

Hints: `np.abs(W).max(axis=0)` gives one value per column, and `np.where(scales == 0, 1.0, scales)` replaces zero scales.

*Type it rather than pasting it: typing is how the shapes and indices sink in.*

In [ ]:
def quantise_int8_reference(W):
    scales = np.abs(W).max(axis=0) / 127
    scales = np.where(scales == 0, 1.0, scales)
    q = np.clip(np.round(W / scales), -127, 127).astype(np.int8)
    return q, scales


def dequantise_reference(q, scales):
    return q.astype(np.float64) * scales

In [ ]:
def quantise_int8(W):
    """(q, scales): q int8 of W's shape; scales (d_out,) = column max |W| / 127, or 1.0 for an all-zero column."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError


def dequantise(q, scales):
    """q as floats times the column scales."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
W_check = np.array([[0.5, -2.0, 0.0], [-1.27, 1.0, 0.0], [0.02, 0.3, 0.0]])
quantise_int8 = practice("quantise_int8", quantise_int8, quantise_int8_reference,
                         cases=[(W_check,), (W_small,), (-W_small,)])
dequantise = practice("dequantise", dequantise, dequantise_reference,
                      cases=[quantise_int8_reference(W_check), quantise_int8_reference(W_small)])
q_check, scales_check = quantise_int8(W_check)
assert q_check.dtype == np.int8, "store the integers with .astype(np.int8)"
print(q_check, scales_check, sep="\n")

**Expected behaviour.** The third column of `W_check` is all zeros, so its scale is 1 and its integers are 0. Each column's largest weight in absolute value maps to $\pm 127$.

Now quantise every adapted matrix of the frozen base, measure the error and the memory, and evaluate the model with the rebuilt weights.

In [ ]:
quantised = {name: quantise_int8(base_frozen[name].data) for name in LORA_TARGETS}
base_int8 = dict(base_frozen)
for name, (q, scales) in quantised.items():
    base_int8[name] = Tensor(dequantise(q, scales))
worst = max(np.abs(base_int8[n].data - base_frozen[n].data).max() / np.abs(base_frozen[n].data).max(axis=0).max()
            for n in LORA_TARGETS)
bytes_float32 = sum(base_frozen[n].data.astype(np.float32).nbytes for n in LORA_TARGETS)
bytes_int8 = sum(q.nbytes + scales.astype(np.float32).nbytes for q, scales in quantised.values())
print(f"largest error relative to the largest weight of its matrix: {worst:.4%}  (bound 1/254 = {1 / 254:.4%})")
print(f"memory of the adapted matrices: float32 {bytes_float32:,} bytes -> int8 + float32 scales {bytes_int8:,} bytes")
print("int8 base:            ", task_accuracy(base_int8, OLD_TASKS))
print("int8 base + adapters: ", task_accuracy(base_int8, OLD_TASKS + ["next"], adapters=adapters))

**What to notice.** Every error stays within the bound, the stored matrices shrink almost fourfold relative to float32, and the accuracy does not move: at 8 bits with per-column scales, the rounding is far smaller than anything these tasks depend on.

### 🧪 Experiment: fewer bits, and QLoRA in miniature

With $b$ bits, symmetric quantisation has $2^{b-1} - 1$ positive levels instead of 127, so each step is larger. The first cell sweeps $b$ and evaluates the base, and the base with the adapters you trained on the **float** base. The second cell does what QLoRA does: it trains a *fresh* adapter on top of the quantised base, so the adapter learns to work with the rounded weights. Change `qlora_bits` to 4 and to 2 and run the second cell again.

In [ ]:
def quantise_bits(W, bits):
    """Symmetric per-column quantisation to 2**(bits - 1) - 1 positive levels, returned already rebuilt as floats."""
    levels = 2 ** (bits - 1) - 1
    scales = np.abs(W).max(axis=0) / levels
    scales = np.where(scales == 0, 1.0, scales)
    return np.clip(np.round(W / scales), -levels, levels) * scales


def quantised_base(bits):
    params = dict(base_frozen)
    for name in LORA_TARGETS:
        params[name] = Tensor(quantise_bits(base_frozen[name].data, bits))
    return params


print("bits  old instructions (base)   with the float-trained adapters: old / next")
for bits in (8, 6, 4, 3, 2):
    q_params = quantised_base(bits)
    old = np.mean(list(task_accuracy(q_params, OLD_TASKS).values()))
    adapted = task_accuracy(q_params, OLD_TASKS + ["next"], adapters=adapters)
    old_adapted = np.mean([adapted[t] for t in OLD_TASKS])
    print(f"{bits:>4}  {old:>24.2f}   {old_adapted:>32.2f} / {adapted['next']:.2f}")

In [ ]:
qlora_bits = 3  # 🧪 try 4 and 2
base_low = quantised_base(qlora_bits)
float_adapter_on_low = task_accuracy(base_low, OLD_TASKS + ["next"], adapters=adapters)
qlora_adapters = init_adapters(base_low, LORA_TARGETS, rank=rank, alpha=2 * rank, seed=0)
start = time.perf_counter()
sft_train(base_low, ["next"] + OLD_TASKS, steps=150, seed=10, lr=5e-3, task_weights=task_weights, adapters=qlora_adapters)
qlora_accuracy = task_accuracy(base_low, OLD_TASKS + ["next"], adapters=qlora_adapters)
print(f"trained a fresh adapter on the {qlora_bits}-bit base in {time.perf_counter() - start:.1f} s")
print(f"{qlora_bits}-bit base + adapters trained on the float base:", float_adapter_on_low)
print(f"{qlora_bits}-bit base + adapters trained on it (QLoRA):    ", qlora_accuracy)

**What to notice.** Down to about 4 bits the base itself hardly changes, but the adapters trained on the float base lose accuracy once the base under them is rounded: they learned a correction for weights that are no longer there. At 3 bits the base starts to slip too, and at 2 bits it collapses. Training the adapter *on the quantised base* recovers the new instruction and much of the old behaviour, because the adapter learns to compensate for the rounding it actually sees. That is the essential idea of QLoRA, here with plain uniform levels instead of 4-bit NormalFloat.

## 5 · Preference optimisation: what the DPO loss rewards

Supervised fine-tuning imitates one target response. **Preference data** instead says which of two responses to the same prompt $x$ is better under a stated rubric: a *chosen* response $y_w$ and a *rejected* one $y_l$. A common RLHF pipeline first fits a reward model to such comparisons and then optimises the policy against it with reinforcement learning. **Direct Preference Optimisation** (DPO) skips the separate reward model and trains the policy $\pi_\theta$ directly, relative to a frozen **reference** policy $\pi_\text{ref}$, usually the SFT model:

$$\mathcal{L}_\text{DPO} = -\log \sigma\!\Big(\beta\big[\underbrace{\log \pi_\theta(y_w \mid x) - \log \pi_\text{ref}(y_w \mid x)}_{\text{chosen: change from reference}} - \underbrace{\big(\log \pi_\theta(y_l \mid x) - \log \pi_\text{ref}(y_l \mid x)\big)}_{\text{rejected: change from reference}}\big]\Big).$$

The quantity $\hat r(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_\text{ref}(y \mid x)}$ acts as an **implicit reward**, and the loss is small when the chosen response's implicit reward exceeds the rejected one's. The coefficient $\beta$ (often around 0.1) sets how strongly the margin is pushed, and so how far the policy may drift from the reference.

For a language model, $\log \pi(y \mid x)$ is the **sum** of the log-probabilities of the response tokens, read off the model's log-softmax at the reply positions, which is exactly the response mask of notebook 02.

In [ ]:
log_probs_demo = np.log(np.array([[[0.7, 0.2, 0.1], [0.1, 0.8, 0.1]]]))  # (sequences, T, vocabulary)
print("log p of the targets [0, 1]:", log_probs_demo[0, 0, 0], "+", log_probs_demo[0, 1, 1],
      "=", log_probs_demo[0, 0, 0] + log_probs_demo[0, 1, 1])

### ✍️ Your turn: `sequence_logprob`

Write `sequence_logprob(log_probs, targets, mask)`.

- `log_probs`: shape `(N, T, V)`, log-probabilities over the vocabulary at each position;
- `targets`: integers of shape `(N, T)`; `mask`: 0/1 floats of shape `(N, T)`;
- return a float array of shape `(N,)`: for each sequence, the sum over positions $t$ of `mask[n, t] * log_probs[n, t, targets[n, t]]`.

Hint: `np.take_along_axis(log_probs, targets[..., None], axis=-1)[..., 0]` picks the log-probability of each target.

*Type it rather than pasting it: typing is how the shapes and indices sink in.*

In [ ]:
def sequence_logprob_reference(log_probs, targets, mask):
    picked = np.take_along_axis(log_probs, np.asarray(targets)[..., None], axis=-1)[..., 0]
    return (picked * mask).sum(axis=-1)

In [ ]:
def sequence_logprob(log_probs, targets, mask):
    """(N,) sums over t of mask[n, t] * log_probs[n, t, targets[n, t]]. Shapes (N, T, V), (N, T), (N, T)."""
    # ✍️ Your code here (about two lines)
    raise NotImplementedError

In [ ]:
raw = np.random.default_rng(2).normal(size=(2, 3, 4))
log_probs_check = raw - np.log(np.exp(raw).sum(axis=-1, keepdims=True))
sequence_logprob = practice("sequence_logprob", sequence_logprob, sequence_logprob_reference,
                            cases=[(log_probs_check, np.array([[0, 1, 2], [3, 3, 0]]), np.array([[0., 1., 1.], [1., 1., 1.]])),
                                   (log_probs_demo, np.array([[0, 1]]), np.array([[1., 1.]])),
                                   (log_probs_check, np.zeros((2, 3), dtype=int), np.zeros((2, 3)))])

### ✍️ Your turn: `dpo_loss`

Write `dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta)`. Each of the first four arguments is an array of shape `(N,)` of sequence log-probabilities. Return the **mean** over the $N$ pairs of $-\log\sigma(\beta\,m)$, where

$$m = (\text{policy\_chosen} - \text{ref\_chosen}) - (\text{policy\_rejected} - \text{ref\_rejected}).$$

Compute $-\log \sigma(z)$ stably as `np.logaddexp(0.0, -z)`, which equals $\log(1 + e^{-z})$ without overflowing for large $|z|$. Return a Python float.

*Type it rather than pasting it: typing is how the shapes and indices sink in.*

In [ ]:
def dpo_loss_reference(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta):
    margin = (np.asarray(policy_chosen) - ref_chosen) - (np.asarray(policy_rejected) - ref_rejected)
    return float(np.mean(np.logaddexp(0.0, -beta * margin)))

In [ ]:
def dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta):
    """Mean over pairs of -log sigmoid(beta * ((pc - rc) - (pr - rr))), computed with np.logaddexp."""
    # ✍️ Your code here (about two lines)
    raise NotImplementedError

In [ ]:
pc, pr, rc, rr = np.array([-3.0, -5.0, -1.0]), np.array([-4.0, -4.5, -9.0]), np.array([-3.5, -5.0, -2.0]), np.array([-3.5, -5.0, -2.0])
dpo_loss = practice("dpo_loss", dpo_loss, dpo_loss_reference,
                    cases=[(pc, pr, rc, rr, 0.1), (rc, rr, rc, rr, 0.5),
                           (np.array([0.0]), np.array([-2000.0]), np.array([0.0]), np.array([0.0]), 1.0),
                           (np.array([-2000.0]), np.array([0.0]), np.array([0.0]), np.array([0.0]), 1.0)])
print("policy equal to the reference:", dpo_loss(rc, rr, rc, rr, 0.1), " ln 2 =", np.log(2))

**Expected behaviour.** When the policy equals the reference, every margin is 0 and the loss is $\ln 2 \approx 0.693$, whatever the log-probabilities are. A margin of $+2000$ gives a loss of almost exactly 0 and a margin of $-2000$ a loss of 2000, with no overflow warning.

Now look at what the loss rewards. Each row starts from the same reference log-probabilities, $-10$ for the chosen and $-12$ for the rejected response, and changes the policy in a different way.

In [ ]:
scenarios = [("policy equals reference", -10, -12), ("chosen up by 2", -8, -12), ("rejected down by 4", -10, -16),
             ("both down, rejected more", -13, -20), ("both up by 2", -8, -10), ("chosen down by 2", -12, -12)]
print(f"{'scenario':<26} {'log p chosen':>12} {'log p rejected':>15} {'loss (beta 0.5)':>16}")
for name, chosen, rejected in scenarios:
    value = dpo_loss(np.array([chosen]), np.array([rejected]), np.array([-10.0]), np.array([-12.0]), 0.5)
    print(f"{name:<26} {chosen:>12} {rejected:>15} {value:>16.3f}")

**What to notice.** DPO rewards the **gap** between the chosen and rejected responses' changes from the reference, not the probability of the chosen response itself. Raising both by the same amount changes nothing. "Both down, rejected more" scores *better* than "chosen up by 2": the probability of the preferred answer fell, yet the loss improved. Real DPO runs do sometimes lower the likelihood of the chosen responses, which is one reason to evaluate the model's actual answers rather than the training loss.

The loss can be computed for our tiny models directly. With LoRA the reference comes for free: it is the same network with the adapters switched off. The adapters were trained with SFT, not with DPO, but the implicit reward still shows which way they moved the model. For `next` prompts, compare the correct answer (chosen) with simply copying the string (rejected); for `reverse` prompts, compare the reversed string with the copy.

In [ ]:
def response_logprobs(params, conversations, adapters=None):
    """Summed log-probability of each conversation's reply under the model."""
    rows = [response_only_targets(encode(render_chat(m)), assistant_flags(m)) for m in conversations]
    inputs, targets, mask = pad_rows(rows)
    return sequence_logprob(gpt_forward(params, inputs, adapters).log_softmax(axis=-1).data, targets, mask)


def as_reply(task, s, reply):
    return [{"role": "user", "content": f"{task} {s}"}, {"role": "assistant", "content": reply}]


BETA = 0.1
for task in ("next", "reverse"):
    chosen = [as_reply(task, s, TASKS[task](s)) for s in EVAL_STRINGS]
    rejected = [as_reply(task, s, s) for s in EVAL_STRINGS]  # the copy
    policy_c, policy_r = response_logprobs(base_frozen, chosen, adapters), response_logprobs(base_frozen, rejected, adapters)
    ref_c, ref_r = response_logprobs(base_frozen, chosen), response_logprobs(base_frozen, rejected)
    reward_c, reward_r = BETA * (policy_c - ref_c), BETA * (policy_r - ref_r)
    print(f"{task:<8} DPO loss {dpo_loss(policy_c, policy_r, ref_c, ref_r, BETA):.3f} | mean implicit reward: "
          f"chosen {reward_c.mean():+.2f}, rejected {reward_r.mean():+.2f} | pairs ranked correctly {np.mean(reward_c > reward_r):.0%}")

**What to notice.** For `next`, the adapters raised the correct answers and lowered the copies relative to the reference, so the DPO loss is far below $\ln 2$ and almost every pair is ranked correctly. For `reverse`, the policy hardly differs from the reference, so the loss stays near $\ln 2$: DPO measures change *relative to the reference*, not absolute quality, even though both models already reverse strings perfectly.

### 🧪 Experiment: a length shortcut in preference data

To see what DPO learns from the *data*, use a policy small enough to inspect. Each prompt has four candidate responses, described by two features: whether the response is correct (0 or 1) and its length (in tens of tokens). Short responses have length 0.2–0.5 and long ones 1.0–2.0. The policy scores each candidate as $\phi(y)^\top w$ and turns the scores into probabilities with a softmax over the four candidates; the reference uses fixed weights $w_\text{ref}$ that mildly prefer correct answers. This is a two-feature scorer over four fixed candidates, not a language model, but the DPO loss is the same.

In the **biased** data, the chosen response is always correct *and long* and the rejected one wrong *and short*, so length and correctness are confounded. In the **mixed** data, a third of the prompts add a pair in which a short correct answer beats a long wrong one. After training, test every prompt on the conflict: does the policy prefer the short correct answer to the long wrong one?

DPO needs $\log \sigma$ inside the tensor engine. Computing `z.sigmoid().log()` overflows to $-\infty$ for large negative $z$, so `log_sigmoid` uses $\log\sigma(z) = z - \log(1 + e^z)$, the second entry of `log_softmax([0, z])`, built from existing operations and checked against numerical gradients first.

In [ ]:
def log_sigmoid(z):
    """log(sigmoid(z)) for a Tensor of shape (N,), stable for large |z|: the second entry of log_softmax([0, z])."""
    return (z.reshape(-1, 1) * np.array([0.0, 1.0])).log_softmax(axis=-1)[:, 1]


z_check = Tensor(np.array([-30.0, -2.0, 0.0, 3.0, 40.0]), requires_grad=True)
log_sigmoid(z_check).sum().backward()
error = relative_error(z_check.grad, numerical_grad(lambda: log_sigmoid(z_check).sum(), z_check))
print(f"log_sigmoid gradient check: relative error {error:.1e}")
assert error < 1e-6

picker = np.random.default_rng(0)
n_prompts = 60
features = np.stack([np.array([[1, picker.uniform(0.2, 0.5)], [1, picker.uniform(1.0, 2.0)],
                               [0, picker.uniform(0.2, 0.5)], [0, picker.uniform(1.0, 2.0)]])
                     for _ in range(n_prompts)])  # (prompt, candidate, feature): 0 correct-short, 1 correct-long, 2 wrong-short, 3 wrong-long
w_ref = np.array([[0.5], [0.2]])


def candidate_logprobs(w, prompts):
    """log pi(candidate | prompt) for every candidate: a log-softmax over the four scores."""
    return (Tensor(features[prompts]) @ w).reshape(len(prompts), 4).log_softmax(axis=-1)


def train_dpo(pairs, beta=1.0, steps=300, lr=0.1):
    prompts, chosen, rejected = (np.array(column) for column in zip(*pairs))
    ref = candidate_logprobs(Tensor(w_ref), prompts).data
    ref_margin = ref[np.arange(len(pairs)), chosen] - ref[np.arange(len(pairs)), rejected]
    w = Tensor(w_ref.copy(), requires_grad=True)
    losses = []
    for _ in range(steps):
        logp = candidate_logprobs(w, prompts)
        margin = logp[np.arange(len(pairs)), chosen] - logp[np.arange(len(pairs)), rejected] - ref_margin
        loss = -log_sigmoid(margin * beta).mean()
        w.grad = None
        loss.backward()
        w.data -= lr * w.grad
        losses.append(float(loss.data))
    return w.data, losses


def prefers_short_correct(w):
    logp = candidate_logprobs(Tensor(w), np.arange(n_prompts)).data
    return float(np.mean(logp[:, 0] > logp[:, 3]))


biased = [(p, 1, 2) for p in range(n_prompts)]
mixed = biased + [(p, 0, 3) for p in range(0, n_prompts, 3)]
results = {"reference": (w_ref, [np.nan])}
for name, pairs in (("biased data", biased), ("mixed data", mixed)):
    results[name] = train_dpo(pairs)
print(f"{'policy':<12} {'weight: correct':>16} {'weight: length':>15} {'DPO loss first -> last':>23}  prefers short correct over long wrong")
for name, (w, losses) in results.items():
    print(f"{name:<12} {w[0, 0]:>16.2f} {w[1, 0]:>15.2f} {losses[0]:>11.3f} -> {losses[-1]:<9.3f}  {prefers_short_correct(w):.0%} of prompts")

In [ ]:
plt.figure(figsize=(6.5, 3.2))
for name in ("biased data", "mixed data"):
    plt.plot(results[name][1], label=name)
plt.axhline(np.log(2), linestyle=":", color="grey")
plt.text(300, np.log(2), " ln 2", va="center", fontsize=8)
plt.xlabel("gradient step")
plt.ylabel("DPO loss")
plt.legend()
plt.tight_layout()
plt.show()

**What to notice.** Both runs start at $\ln 2$, because the policy starts equal to the reference, and both drive the loss down. Trained on biased data, the policy puts as much weight on length as on correctness, and on the conflict it now prefers the long wrong answer for many prompts, although the reference preferred the short correct one on every prompt. Nothing in the loss distinguishes a real quality signal from a feature that merely accompanies it. A third of the prompts with counter-examples is enough to move the weight back onto correctness. Write the rubric first, check the pairs for shortcuts (length, format, stock phrases), and evaluate on held-out comparisons that separate them.

## Put it together

The final cell re-checks the claims of this notebook. If you changed `rank`, `replay_share` or `qlora_bits` in the 🧪 cells, some measured floors may no longer hold; the message says which.

In [ ]:
# LoRA: identical at the start, a frozen base, and a modest number of trainable parameters.
fresh = init_adapters(base_frozen, LORA_TARGETS, rank=rank, alpha=2 * rank, seed=1)
assert np.array_equal(gpt_forward(base_frozen, probe_ids, fresh).data, gpt_forward(base_frozen, probe_ids).data)
assert changed_weights(before, {name: t.data for name, t in base_frozen.items()}) == [], "a frozen weight changed"
n_adapter = sum(A.data.size + B.data.size for A, B, _ in adapters.values())
assert n_adapter == lora_parameter_count([instruct[name].shape for name in LORA_TARGETS], rank)
assert n_adapter < 0.25 * n_base

# The new instruction is learned; with replay, the old ones are mostly kept.
assert accuracy_base["next"] <= 0.1 and accuracy_adapted["next"] >= 0.9, "the adapter did not learn 'next'"
old_mean = np.mean([accuracy_adapted[t] for t in OLD_TASKS])
assert old_mean >= 0.75, f"old instructions fell to {old_mean:.0%}; did you set replay_share to 0?"

# Merging gives the same outputs up to rounding.
assert np.allclose(logits_merged, logits_adapter, atol=1e-9) and same_answers

# int8: every error within half a step, and no loss of accuracy beyond one prompt per instruction.
for name, (q, scales) in quantised.items():
    assert q.dtype == np.int8 and np.all(np.abs(dequantise(q, scales) - base_frozen[name].data) <= scales / 2 + 1e-12)
adapted_int8 = task_accuracy(base_int8, OLD_TASKS + ["next"], adapters=adapters)
assert all(abs(adapted_int8[t] - accuracy_adapted[t]) <= 0.025 + 1e-9 for t in adapted_int8)

# QLoRA in miniature: training on the quantised base beats reusing float-trained adapters on it.
assert qlora_accuracy["next"] >= 0.9 and qlora_accuracy["next"] >= float_adapter_on_low["next"] + 0.1

# DPO: ln 2 at the reference, a falling loss, and a shortcut that counter-examples remove.
assert np.isclose(dpo_loss(rc, rr, rc, rr, 0.1), np.log(2))
assert results["biased data"][1][-1] < results["biased data"][1][0] and results["mixed data"][1][-1] < results["mixed data"][1][0]
assert prefers_short_correct(w_ref) == 1.0 and prefers_short_correct(results["biased data"][0]) <= 0.75
assert prefers_short_correct(results["mixed data"][0]) >= 0.95
print("✅ All checks passed.")
print(f"rank {rank}: {n_adapter:,} trainable parameters ({n_adapter / n_base:.0%} of the base); "
      f"'next' {accuracy_base['next']:.0%} -> {accuracy_adapted['next']:.0%}; old instructions {old_mean:.0%} on average")

## Exercises

The first has a check; the others are questions to answer in a markdown cell of your own, with a little code where it helps.

### Exercise 1 ✍️ `adapter_training_memory`: why QLoRA fits on one accelerator

Write `adapter_training_memory(n_base, n_adapter, base_bytes, adapter_bytes)`, a rough estimate in bytes of the memory for **weights, gradients and optimiser state** when training adapters on a frozen base: the base weights (`n_base * base_bytes`), plus, for the adapter, its weights, its gradients and Adam's two moment estimates, four arrays of `n_adapter * adapter_bytes` each. Activations are left out, which a real estimate must not do.

Then compare a 7-billion-parameter base stored in bfloat16 (2 bytes) and in 4 bits (0.5 bytes), each with 40 million adapter parameters in float32 (4 bytes), against full fine-tuning in the mixed-precision setting of Base Model Training, about 16 bytes per parameter.

In [ ]:
def adapter_training_memory_reference(n_base, n_adapter, base_bytes, adapter_bytes):
    return n_base * base_bytes + 4 * n_adapter * adapter_bytes

In [ ]:
def adapter_training_memory(n_base, n_adapter, base_bytes, adapter_bytes):
    """n_base * base_bytes + 4 * n_adapter * adapter_bytes (adapter weights, gradients and two Adam moments)."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
adapter_training_memory = practice("adapter_training_memory", adapter_training_memory, adapter_training_memory_reference,
                                   cases=[(7e9, 4e7, 2, 4), (7e9, 4e7, 0.5, 4), (100, 10, 1, 4)])
for label, base_bytes in (("bfloat16 base", 2), ("4-bit base", 0.5)):
    print(f"{label}: {adapter_training_memory(7e9, 4e7, base_bytes, 4) / 1e9:.1f} GB")
print(f"full fine-tuning at about 16 bytes per parameter: {7e9 * 16 / 1e9:.0f} GB")

### Exercise 2 (question): which factor starts at zero?

Suppose you initialised $A = 0$ and $B$ at random instead. Using $\partial \mathcal{L} / \partial B = (xA)^\top g$ and $\partial \mathcal{L} / \partial A = x^\top g B^\top$, decide whether training could still start. What if both were zero? Check your answer with one `backward()` through `lora_forward` on small `Tensor`s.

### Exercise 3 (question): the role of $\beta$

Re-run `train_dpo(biased, beta=...)` with `beta` of 0.1, 1 and 10, and compare the learned weights and the loss curves. Why does a larger $\beta$ make the loss fall faster for the same weights? In real DPO, what does $\beta$ control about the distance between the trained policy and the reference, and what risk does a policy run with no limit on that distance take with *deterministic* preferences such as ours?

### Exercise 4 (question): write a rubric for the toy tasks

Write a three-point rubric for judging two answers to `reverse abc`, and three preference pairs that follow it. Then name one shortcut a policy could learn from your pairs (format, length or a repeated phrase) and one held-out comparison that would reveal it.

## Common errors and debugging

- **Initialising both LoRA factors at random.** The adapted model no longer starts at the base, and the first steps undo what pretraining learned. Start $B$ (the output-side factor) at zero and assert identical logits before training.
- **Forgetting to freeze the base.** If base weights are in the optimiser, or require gradients, they move. Snapshot the base and compare bit for bit after training.
- **Losing the scale.** Changing the rank without the $\alpha / r$ scale changes the size of every update; keep $\alpha$ and $r$ together in the saved adapter configuration.
- **Merging and forgetting.** A merged model is a new artefact; evaluate it, especially if it was merged into a quantised base or quantised afterwards.
- **One scale for a whole matrix.** A single large weight wastes the quantisation levels of every column; use per-column (or per-block) scales, and give all-zero columns a non-zero scale.
- **Adapters trained on a different base.** An adapter is a correction to specific weights; on a re-quantised or updated base it can fail, as the experiment showed. Record the exact base revision with every adapter.
- **Averaging response log-probabilities in DPO.** The standard objective sums token log-probabilities; averaging changes what length does to the loss. Whichever you choose, use it for the policy and the reference alike.
- **`log(sigmoid(z))` computed naively.** It overflows for large negative margins; use `np.logaddexp` or a log-softmax.

## Summary

- **LoRA** adds $\frac{\alpha}{r}(xA)B$ beside a frozen $W$; $B$ starts at zero so training starts at the base; an adapter has $r(d_\text{in} + d_\text{out})$ parameters per matrix, a tiny fraction for large matrices.
- **Verify, do not assume:** identical logits at the start, an unchanged base afterwards (bit for bit), and merged outputs equal up to rounding.
- **Quantisation** stores the frozen base in few bits with per-column scales; at 8 bits it cost nothing here. QLoRA trains the adapter on top of the quantised base, which recovers what reusing a float-trained adapter loses.
- **DPO** pushes the chosen response's log-probability ratio to the reference above the rejected one's; it rewards a gap, not the absolute probability of good answers, and it learns whatever separates the chosen from the rejected responses, including shortcuts such as length.

## Next

In **notebook 04** you build a TF-IDF retriever and see when retrieval beats fine-tuning, write an evaluation suite with exact match, per-category scores and a regression check, use it to compare the instruction model with adapted versions (including one trained on `next` alone), and assemble the release bundle for the capstone. Notebook 04 re-creates the instruction model with the same recipe, so you can start it directly.